# 01b · Encoder-equivalent input view
Adds `text_enc` / `context_enc` to data/processed/*.jsonl: the comment and caption **as the
XLM-R encoder saw them** (MAX_LEN 256, `only_second`, comment capped at 248 tokens — thesis §4.3).
Every LLM prompt uses these fields, so encoder and LLM receive the same information.

In [1]:
import sys, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"
MAX_LEN, COMMENT_CAP = 256, 248                 # thesis §4.3.1

tok = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")
SPECIAL = tok.num_special_tokens_to_add(pair=True)   # <s> A </s></s> B </s> = 4
print(type(tok).__name__, "| fast:", tok.is_fast, "| special tokens for a pair:", SPECIAL)

e:\sinhala-sarcasm-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
e:\sinhala-sarcasm-llm\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--FacebookAI--xlm-roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To suppor

XLMRobertaTokenizer | fast: True | special tokens for a pair: 4


In [6]:
def cut(texts, budgets):
    """Cut each text to its token budget, at the character offset of the last kept token."""
    enc = tok(list(texts), add_special_tokens=False, return_offsets_mapping=True)
    out, n_tokens = [], []
    for text, ids, offs, b in zip(texts, enc["input_ids"], enc["offset_mapping"], budgets):
        n_tokens.append(len(ids))
        if len(ids) <= b:
            out.append(text)
        elif b <= 0:
            out.append("")
        else:
            out.append(text[: offs[b - 1][1]])
    return out, np.array(n_tokens)

def pair_lengths(comments, contexts):
    return np.array([len(x) for x in tok(list(comments), list(contexts))["input_ids"]])

def encoder_view(df):
    text_enc, n_comment = cut(df.text, [COMMENT_CAP] * len(df))
    n_comment_kept = np.minimum(n_comment, COMMENT_CAP)
    budgets = (MAX_LEN - SPECIAL - n_comment_kept).astype(int)
    context_enc, n_context = cut(df.context_text, budgets)

    # Repair: a cut inside a word can re-tokenise one token longer -> shrink that caption and re-cut
    text_enc, context_enc = np.array(text_enc, dtype=object), np.array(context_enc, dtype=object)
    repaired = 0
    for _ in range(10):
        lens = pair_lengths(text_enc, context_enc)
        over = np.where(lens > MAX_LEN)[0]
        if len(over) == 0:
            break
        repaired += len(over)
        budgets[over] -= (lens[over] - MAX_LEN)
        new_ctx, _ = cut(df.context_text.to_numpy()[over], budgets[over])
        context_enc[over] = new_ctx
    else:
        raise RuntimeError("some pairs still exceed MAX_LEN after 10 repair passes")

    return df.assign(text_enc=list(text_enc), context_enc=list(context_enc),
                     _n_comment=n_comment, _n_context=n_context, _budget=budgets), repaired

splits = {}
for s in ("train", "val", "test"):
    df = pd.read_json(PROC_DIR / f"{s}.jsonl", lines=True, dtype=False)
    splits[s], n_rep = encoder_view(df)
    print(f"{s:5s} done ({len(df):,} rows) · boundary repairs: {n_rep}")

train done (32,582 rows) · boundary repairs: 43
val   done (6,938 rows) · boundary repairs: 10
test  done (6,875 rows) · boundary repairs: 8


In [7]:
tr = splits["train"]
pair_fits    = (tr._n_comment + tr._n_context + SPECIAL) <= MAX_LEN
comment_cut  = tr._n_comment > COMMENT_CAP
check = pd.DataFrame({
    "thesis Table 4.2": [99.47, 82.5, 0.53],
    "this tokenizer":   [100 * (~comment_cut).mean(), 100 * pair_fits.mean(), 100 * comment_cut.mean()],
}, index=["comment-only coverage %", "comment–context pair coverage %", "comment-side truncation %"])
check["diff"] = check["this tokenizer"] - check["thesis Table 4.2"]
display(check.round(2))

# The truncated pair must re-tokenise to <= MAX_LEN, and match the encoder's own only_second truncation
sample = tr.sample(2000, random_state=1)
full = tok(list(sample.text_enc), list(sample.context_text), truncation="only_second", max_length=MAX_LEN)["input_ids"]
mine = tok(list(sample.text_enc), list(sample.context_enc))["input_ids"]
print("max length after cut :", max(map(len, mine)), "(must be <= 256)")
print("identical to tokenizer's own only_second truncation:",
      f"{np.mean([a == b for a, b in zip(full, mine)]):.1%} of 2,000 rows")

,thesis Table 4.2,this tokenizer,diff
comment-only coverage %,99.47,99.45,-0.02
comment–context pair coverage %,82.50,82.45,-0.05
comment-side truncation %,0.53,0.55,0.02


max length after cut : 256 (must be <= 256)
identical to tokenizer's own only_second truncation: 99.7% of 2,000 rows


In [8]:
CHARS_PER_GPT_TOKEN = 1.9          # measured in notebook 03, Cell 9
rows = []
for s, df in splits.items():
    for script, g in df.groupby("script_type"):
        before = (g.text.str.len() + g.context_text.str.len()).mean()
        after  = (g.text_enc.str.len() + g.context_enc.str.len()).mean()
        rows.append({"split": s, "script": script, "chars before": before, "chars after": after,
                     "≈ gpt-oss tokens after": after / CHARS_PER_GPT_TOKEN,
                     "captions cut %": 100 * (g.context_enc.str.len() < g.context_text.str.len()).mean()})
display(pd.DataFrame(rows).round(1))

,split,script,chars before,chars after,≈ gpt-oss tokens after,captions cut %
0,train,latin,431.9,294.8,155.2,13.8
1,train,mixed,701.0,427.7,225.1,26.1
2,train,sinhala,493.7,326.9,172.0,17.1
3,val,latin,411.1,276.1,145.3,14.3
4,val,mixed,701.4,413.1,217.4,26.1
5,val,sinhala,519.4,357.2,188.0,20.5
6,test,latin,354.4,271.2,142.7,13.1
7,test,mixed,889.4,435.2,229.0,27.2
8,test,sinhala,475.5,314.6,165.6,17.3


In [9]:
FIELDS = ["id", "post_id", "post_group", "text", "context_text", "text_enc", "context_enc",
          "script_type", "sentiment", "is_sarcastic", "split"]
for s, df in splits.items():
    with open(PROC_DIR / f"{s}.jsonl", "w", encoding="utf-8") as f:
        for rec in df[FIELDS].to_dict("records"):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"rewrote data/processed/{s}.jsonl with text_enc / context_enc")

print(f"""
## {datetime.date.today()} · Encoder-equivalent LLM input (notebook 01b)
- Decision: every comment–caption pair shown to the LLM (target, retrieved, few-shot) is truncated exactly as the
  XLM-R encoder saw it (MAX_LEN 256, only_second, comment cap 248; thesis §4.3). Controlled comparison:
  encoder and LLM receive the same information; context effects are tested separately by A1.
- Coverage vs thesis Table 4.2: pair {check.loc['comment–context pair coverage %','this tokenizer']:.2f}% (thesis 82.5),
  comment-cut {check.loc['comment-side truncation %','this tokenizer']:.2f}% (thesis 0.53).
- Approximation: public tokenizer lacks the thesis's 213 added emoji tokens (±1–2 tokens on affected rows).
""")

rewrote data/processed/train.jsonl with text_enc / context_enc
rewrote data/processed/val.jsonl with text_enc / context_enc
rewrote data/processed/test.jsonl with text_enc / context_enc

## 2026-11-21 · Encoder-equivalent LLM input (notebook 01b)
- Decision: every comment–caption pair shown to the LLM (target, retrieved, few-shot) is truncated exactly as the
  XLM-R encoder saw it (MAX_LEN 256, only_second, comment cap 248; thesis §4.3). Controlled comparison:
  encoder and LLM receive the same information; context effects are tested separately by A1.
- Coverage vs thesis Table 4.2: pair 82.45% (thesis 82.5),
  comment-cut 0.55% (thesis 0.53).
- Approximation: public tokenizer lacks the thesis's 213 added emoji tokens (±1–2 tokens on affected rows).

